# CSE — Sentiment Analysis Baseline

Analysis of news sentiment and its relationship to market returns.

**Covers:**
1. VADER score distribution across LBO corpus
2. Sentiment label breakdown
3. Event study — market return on high-sentiment vs low-sentiment days
4. Sentiment as a feature — correlation with next-day market return

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
sent = pd.read_csv('../data/processed/news/unified_sentiment.csv', parse_dates=['date'])
df   = pd.read_parquet('../data/published/cse_unified.parquet')
df['date'] = pd.to_datetime(df['date'])

print(f'Sentiment corpus: {len(sent)} articles')
print(f'Sentiment date range: {sent["date"].dropna().min().date()} to {sent["date"].dropna().max().date()}')
print(f'Label distribution:')
print(sent['vader_label'].value_counts())

## 1. VADER Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(sent['vader_score'], bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(0.05,  color='green',  lw=1.5, linestyle='--', label='positive threshold')
axes[0].axvline(-0.05, color='red',    lw=1.5, linestyle='--', label='negative threshold')
axes[0].set_xlabel('VADER compound score')
axes[0].set_ylabel('Count')
axes[0].set_title('VADER Score Distribution')
axes[0].legend()

label_counts = sent['vader_label'].value_counts()
colors = {'positive': 'green', 'neutral': 'grey', 'negative': 'red'}
bar_colors = [colors.get(l, 'blue') for l in label_counts.index]
axes[1].bar(label_counts.index, label_counts.values, color=bar_colors)
axes[1].set_ylabel('Count')
axes[1].set_title('Sentiment Label Distribution')

plt.tight_layout()
plt.show()

## 2. Sentiment by Source

In [ ]:
source_sent = sent.groupby('source')['vader_score'].describe().round(3)
print('Sentiment stats by source:')
print(source_sent.to_string())

## 3. Daily Market Sentiment Time Series

In [ ]:
daily_sent = (
    sent.dropna(subset=['date'])
    .groupby(sent['date'].dt.normalize())['vader_score']
    .mean()
    .rename('market_sentiment')
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(daily_sent.index, daily_sent.values, alpha=0.4,
                where=daily_sent.values >= 0, color='green', label='Positive')
ax.fill_between(daily_sent.index, daily_sent.values, alpha=0.4,
                where=daily_sent.values < 0,  color='red',   label='Negative')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Mean VADER score')
ax.set_title('Daily Market Sentiment (LBO news)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Event Study — Sentiment vs Next-Day Market Return

In [ ]:
# Average market return per day
mkt_ret = (
    df[df['is_trading_day']]
    .groupby('date')['return_1d']
    .mean()
    .rename('mkt_return')
)

event = pd.concat([daily_sent, mkt_ret], axis=1).dropna()
# Use next-day market return
event['next_mkt_return'] = event['mkt_return'].shift(-1)
event = event.dropna()

print(f'Overlapping sentiment+return days: {len(event)}')

# Bucket into terciles
event['sent_tercile'] = pd.qcut(event['market_sentiment'], q=3,
                                 labels=['Low (neg)','Mid','High (pos)'])

tercile_ret = event.groupby('sent_tercile', observed=True)['next_mkt_return'].agg(['mean','std','count'])
print('\nNext-day market return by sentiment tercile:')
print(tercile_ret.round(5).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
tercile_ret['mean'].plot(kind='bar', ax=ax,
    color=['red','grey','green'], yerr=tercile_ret['std']/np.sqrt(tercile_ret['count']))
ax.set_ylabel('Mean next-day market return')
ax.set_title('Next-Day Market Return by Sentiment Tercile')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 5. Pearson Correlation: Sentiment vs Next-Day Return

In [ ]:
corr = event[['market_sentiment','next_mkt_return']].corr().iloc[0,1]
print(f'Pearson correlation (sentiment → next-day mkt return): {corr:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(event['market_sentiment'], event['next_mkt_return'],
           alpha=0.4, s=15, color='steelblue')
m, b = np.polyfit(event['market_sentiment'], event['next_mkt_return'], 1)
x_line = np.linspace(event['market_sentiment'].min(), event['market_sentiment'].max(), 100)
ax.plot(x_line, m*x_line + b, color='red', lw=1.5, label=f'y = {m:.4f}x + {b:.4f}')
ax.set_xlabel('Daily market sentiment (VADER mean)')
ax.set_ylabel('Next-day market return')
ax.set_title(f'Sentiment vs Next-Day Return  (r={corr:.3f})')
ax.legend()
plt.tight_layout()
plt.show()